<a href="https://colab.research.google.com/github/AgustinaQuadro/AgustinaQuadro.github.io/blob/main/notebooks/limpieza_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Limpieza y agregación de los datos — TP Subte

Este notebook transforma los zips originales de SBASE (unos 12 GB descomprimidos) en archivos livianos con los pasajeros **por fecha, hora, línea y estación**, que son los que usa el notebook del análisis.

Para cada año:
1. Abre el zip original sin descomprimirlo en disco y recorre cada CSV fila por fila.
2. Corrige la fecha según cómo viene escrita en ese año.
3. Unifica los nombres de las líneas y de las estaciones.
4. Suma los pasajeros por fecha, hora, línea y estación.
5. Guarda el resultado en `data/processed/por_anio/molinetes_AAAA.csv.gz` y lo sube al repositorio.

Los problemas de los datos originales que motivan cada paso están en `data/README.md` y en el notebook `exploracion/inspeccion_datos.ipynb`.

**Antes de correrlo:** los zips tienen que estar en `Mi unidad/TP-Subte/zips/` y el secreto `GITHUB_TOKEN` cargado en Colab (ícono de la llave), con acceso activado para este notebook.

**Tiempo:** unos 30 a 45 minutos. Cada año terminado se copia a Drive (`Mi unidad/TP-Subte/procesados/`): si Colab se desconecta, al volver a correrlo los años hechos se recuperan de ahí.

🤖 **Asistencia de IA:** sí, en todas las celdas de código.
**Prompt usado:** «(completar con los prompts reales)»
**Qué revisó el grupo:** (completar)

## 1. Configuración

Lo único que hay que tocar. El email tiene que ser el de la cuenta de GitHub de quien commitea: así GitHub le asigna el commit.

In [1]:
import os
import io
import shutil
import zipfile
import subprocess
import datetime as dt
import pandas as pd

GIT_USER_NAME = "AgustinaQuadro"
GIT_USER_EMAIL = "agusyeuge@gmail.com"

REPO_URL = "https://github.com/julia-iza/Trabajo-Practico-Laboratorio.git"
REPO_DIR = "/content/Trabajo-Practico-Laboratorio"
CARPETA_ZIPS = "/content/drive/MyDrive/TP-Subte/zips"
CARPETA_RESPALDO = "/content/drive/MyDrive/TP-Subte/procesados"   # copia de seguridad en Drive

ANIOS = list(range(2015, 2027))

## 2. Drive y repositorio

Monta Drive (donde están los zips) y clona el repositorio, o trae los últimos cambios si ya estaba clonado en esta sesión.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!git config --global user.name "{GIT_USER_NAME}"
!git config --global user.email "{GIT_USER_EMAIL}"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --rebase
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

Mounted at /content/drive
Cloning into '/content/Trabajo-Practico-Laboratorio'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 35 (delta 10), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 88.76 KiB | 4.23 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/Trabajo-Practico-Laboratorio


## 3. Parámetros de la limpieza

- Dónde se guardan los resultados.
- `ULTIMA_FECHA`: último día con datos reales; los registros posteriores se descartan.
- `TAM_BLOQUE`: cuántas filas se juntan antes de agruparlas. Leer todo junto no entra en la memoria de Colab.
- `RENOMBRAR`: algunos años llaman distinto a las mismas columnas.
- `FORMATO_FECHA`: cómo viene escrita la fecha en cada archivo anual, según la inspección. Desde 2022 hay un archivo por mes y la fecha se resuelve con el mes del nombre del archivo.
- `CONTROLES`: valores calculados antes, para comparar al final.

In [3]:
CARPETA_SALIDA = os.path.join("data", "processed", "por_anio")
ARCHIVO_RESUMEN = os.path.join("data", "processed", "resumen_limpieza.csv")
ARCHIVO_ESTACIONES = os.path.join("data", "processed", "estaciones_por_anio.csv")

ANIOS = list(range(2015, 2027))
ULTIMA_FECHA = pd.Timestamp("2026-08-31")   # último día con datos reales
TAM_BLOQUE = 500_000                        # filas que se leen antes de agrupar

COLUMNAS_PAX = ["pax_pagos", "pax_pases_pagos", "pax_franq", "total"]

# Algunos años usan otros nombres para las mismas columnas.
RENOMBRAR = {"pax_freq": "pax_franq", "pax_total": "total"}

# Cómo viene escrita la fecha en los archivos anuales (hallazgos de la inspección):
#   "iso"       -> 2019-01-31, siempre año-mes-día
#   "dia/mes"   -> 31/01/2016, siempre día/mes/año (el archivo NO está ordenado por fecha)
#   "secuencia" -> mezcla día/mes, mes/día e ISO invertida; el archivo SÍ está ordenado,
#                  así que cada fecha ambigua se resuelve mirando la fecha anterior.
# Desde 2022 hay un CSV por mes (por ejemplo 202508_PAX15min-ABC.csv): la fecha se
# resuelve con el año y el mes que figuran al principio del nombre del archivo.
FORMATO_FECHA = {
    2015: "iso", 2018: "iso", 2019: "iso",
    2016: "dia/mes", 2017: "dia/mes",
    2020: "secuencia", 2021: "secuencia",
}

# Valores de control calculados antes con 2019, 2020, 2021, 2025 y 2026 (método anterior).
CONTROLES = {
    "Pasajeros totales": 855_064_366,
    "Fechas distintas": 1_700,
    "Promedio diario feb-2020": 767_900,
    "Promedio diario abr-2020": 28_400,
    "Promedio diario abr-2019": 965_900,
    "Promedio diario ago-2025": 610_600,
    "Promedio diario ago-2026": 586_500,
    "Total ene-2019": 22_849_359,
    "Total abr-2020": 852_910,
}

## 4. Corrección de las fechas

Una fecha como `3/4/2020` puede ser 3 de abril (día/mes) o 4 de marzo (mes/día), y en `2021-01-02` el día y el mes pueden estar invertidos. Por eso:

- `fecha_valida` arma una fecha y devuelve `None` si es imposible (por ejemplo, mes 13).
- `lecturas_posibles` devuelve las dos lecturas de un texto: la **normal** (`DD/MM/AAAA` o `AAAA-MM-DD`) y la **invertida** (`MM/DD/AAAA` o `AAAA-DD-MM`).
- `mas_cercana` elige, entre las lecturas posibles, la más cercana a una fecha de referencia.
- `resolver_por_mes` se usa desde 2022: como cada archivo es de un mes (figura en el nombre, por ejemplo `202508_...`), elige la lectura que cae en ese mes.

Para 2020 y 2021 la referencia es el día siguiente a la fecha de la fila anterior, porque esos archivos están ordenados por fecha. Es una inferencia, no un dato oficial.

In [4]:
def fecha_valida(anio, mes, dia):
    """Devuelve la fecha si existe, o None si es imposible (por ejemplo, mes 13)."""
    try:
        return dt.date(anio, mes, dia)
    except ValueError:
        return None


def lecturas_posibles(texto):
    """
    Devuelve las dos lecturas de una fecha: [normal, invertida].
    Con guiones: normal = AAAA-MM-DD, invertida = AAAA-DD-MM.
    Con barras:  normal = DD/MM/AAAA, invertida = MM/DD/AAAA.
    Una lectura imposible queda como None.
    """
    texto = texto.strip().strip('"').split(" ")[0]   # por si viene con hora
    try:
        if "-" in texto:
            a, b, c = [int(x) for x in texto.split("-")[:3]]
            return [fecha_valida(a, b, c), fecha_valida(a, c, b)]
        if "/" in texto:
            a, b, c = [int(x) for x in texto.split("/")[:3]]
            return [fecha_valida(c, b, a), fecha_valida(c, a, b)]
    except ValueError:
        pass
    return [None, None]


def mas_cercana(candidatas, objetivo):
    """De las fechas posibles, la más cercana al objetivo."""
    validas = [f for f in candidatas if f is not None]
    if not validas:
        return None
    return min(validas, key=lambda f: abs((f - objetivo).days))


def resolver_por_mes(texto, anio, mes):
    """
    Archivos mensuales: se queda con la lectura que cae en el mes del archivo.
    Devuelve (fecha, fuera_de_mes). Si ninguna lectura cae en el mes, usa la más
    cercana a ese mes y lo marca para el resumen.
    """
    candidatas = [f for f in lecturas_posibles(texto) if f is not None]
    for f in candidatas:
        if f.year == anio and f.month == mes:
            return f, False
    return mas_cercana(candidatas, dt.date(anio, mes, 15)), True

## 5. Lectura de los archivos línea por línea

Los CSV no se pueden leer con un `pd.read_csv` directo porque tienen problemas que varían según el año:

- la primera línea trae un BOM (un carácter invisible) en algunos años;
- algunas líneas están en codificación latin-1 dentro de archivos UTF-8 (estaciones con acentos o eñe);
- desde 2022, cada línea completa viene entre comillas y a veces con `;;;;;;;;;` de más al final;
- en 2016, en cambio, cada campo tiene sus propias comillas, que `pandas` entiende solo.

`leer_lineas` resuelve esto y entrega cada línea limpia.

In [5]:
def leer_lineas(zf, nombre):
    """
    Lee el CSV línea por línea dentro del zip.
    - Saca el BOM de la primera línea.
    - Si una línea no es UTF-8 válido, la lee como latin-1 (pasa en varios años).
    - Si la línea ENTERA está entre comillas (2023 a 2026), saca esas comillas y lo
      que sobre después, como los ';;;;;;;;;' de marzo de 2025. Si cada campo tiene
      sus propias comillas (2016), la deja igual: pandas las entiende solo.
    """
    primera = True
    with zf.open(nombre) as archivo:
        for crudo in archivo:
            try:
                linea = crudo.decode("utf-8-sig" if primera else "utf-8")
            except UnicodeDecodeError:
                linea = crudo.decode("latin-1")
            primera = False
            linea = linea.strip()
            if linea.startswith('"'):
                ultima_comilla = linea.rfind('"')
                interior = linea[1:ultima_comilla]
                if ultima_comilla > 0 and '"' not in interior:
                    linea = interior
            yield linea

## 6. Unificación de líneas y estaciones

El mismo nombre aparece escrito de varias formas según el año:

- Línea: `A` (2016), `LINEA_A` (2015 y 2017) y `LineaA` (el resto) pasan a ser `LineaA`.
- Estación: `Plaza Miserere` y `PLAZA MISERERE` pasan a `PLAZA MISERERE`; `Constitución`, a `CONSTITUCION`; `Callao.B`, a `CALLAO` (la línea ya está en su propia columna).

El archivo `data/processed/estaciones_por_anio.csv` guarda cómo venía escrita cada estación y a qué nombre se llevó, para revisar si quedó algún caso sin unificar.

In [6]:
def normalizar_linea(serie):
    """'A', 'LINEA_A' y 'LineaA' pasan a ser 'LineaA'. 'PM' y 'LineaPM' pasan a 'LineaPM'."""
    letra = serie.str.upper().str.replace("LINEA", "", regex=False)
    letra = letra.str.replace("_", "", regex=False).str.strip()
    return "Linea" + letra


ACENTOS = {"Á": "A", "É": "E", "Í": "I", "Ó": "O", "Ú": "U", "Ü": "U", "Ñ": "N"}


def normalizar_estacion(serie):
    """
    Unifica cómo se escribe una estación entre años:
    'Plaza Miserere' y 'PLAZA MISERERE' -> 'PLAZA MISERERE'
    'Constitución' -> 'CONSTITUCION'
    'Callao.B' -> 'CALLAO' (la línea ya está en su propia columna)
    """
    nombre = serie.str.upper().str.strip()
    for con_acento, sin_acento in ACENTOS.items():
        nombre = nombre.str.replace(con_acento, sin_acento, regex=False)
    nombre = nombre.str.replace(r"\.[A-Z]{1,2}$", "", regex=True)   # sufijo de línea
    nombre = nombre.str.replace(".", " ", regex=False)
    nombre = nombre.str.split().str.join(" ")                       # espacios dobles
    return nombre

## 7. Procesamiento de un CSV

- `agrupar_bloque` convierte un bloque de líneas en DataFrame, pasa los pasajeros a número, la fecha a `datetime`, saca la hora de la columna `desde`, unifica nombres y suma por fecha, hora, línea y estación.
- `procesar_csv` recorre un CSV completo: detecta el separador, ubica la columna de fecha, corrige cada fecha con el método que corresponde al año y va juntando bloques. Cada texto de fecha se resuelve una sola vez y se guarda, para no repetir el cálculo en millones de filas.

También cuenta las filas sin fecha y las que tienen una fecha fuera del mes del archivo, que quedan en `data/processed/resumen_limpieza.csv`.

In [7]:
def agrupar_bloque(buffer, sep, columnas):
    """Convierte un bloque de líneas en DataFrame y lo agrupa por fecha, hora, línea y estación."""
    df = pd.read_csv(io.StringIO("\n".join(buffer)), sep=sep, names=columnas,
                     header=None, dtype=str, on_bad_lines="skip")
    for columna in COLUMNAS_PAX:
        df[columna] = pd.to_numeric(df[columna], errors="coerce")
    df["fecha"] = pd.to_datetime(df["fecha"], format="%Y-%m-%d", errors="coerce")
    df["hora"] = pd.to_numeric(df["desde"].str.split(":").str[0], errors="coerce")
    df = df[df["fecha"].notna() & df["linea"].notna() & df["estacion"].notna()].copy()

    df["estacion_original"] = df["estacion"].str.strip()
    df["linea"] = normalizar_linea(df["linea"])
    df["estacion"] = normalizar_estacion(df["estacion"])

    agregado = df.groupby(["fecha", "hora", "linea", "estacion"], as_index=False)[COLUMNAS_PAX].sum()
    nombres = df.groupby(["linea", "estacion", "estacion_original"], as_index=False)["total"].sum()
    return agregado, nombres


def procesar_csv(zf, nombre, anio):
    """Recorre un CSV, corrige las fechas y devuelve (agregado, nombres, fila_resumen)."""
    lineas = leer_lineas(zf, nombre)
    encabezado = next(lineas, None)
    if encabezado is None:
        return None

    sep = ";" if encabezado.count(";") > encabezado.count(",") else ","
    columnas = [c.strip().strip('"').lower() for c in encabezado.split(sep)]
    columnas = [RENOMBRAR.get(c, c) for c in columnas]
    pos_fecha = columnas.index("fecha")

    nombre_base = os.path.basename(nombre)
    if anio >= 2022:
        metodo = "mes del archivo"
        anio_archivo, mes_archivo = int(nombre_base[:4]), int(nombre_base[4:6])
    else:
        metodo = FORMATO_FECHA[anio]

    cuenta = {"filas": 0, "filas_sin_columnas": 0, "filas_sin_fecha": 0, "filas_fuera_de_mes": 0}
    ya_resueltas = {}          # texto de fecha -> (fecha en ISO, fuera de mes)
    ultimo_texto, ultima_iso, anterior = None, "", None
    bloques, buffer = [], []

    for linea in lineas:
        partes = linea.split(sep, pos_fecha + 1)
        if len(partes) <= pos_fecha:          # línea vacía o rota
            cuenta["filas_sin_columnas"] += 1
            continue
        cuenta["filas"] += 1
        texto = partes[pos_fecha].strip().strip('"')

        if texto == "":
            iso = ""
        elif metodo == "secuencia":
            # El orden importa: solo se recalcula cuando el texto de la fecha cambia.
            if texto != ultimo_texto:
                if anterior is None:
                    f = lecturas_posibles(texto)[0] or lecturas_posibles(texto)[1]
                else:
                    f = mas_cercana(lecturas_posibles(texto), anterior + dt.timedelta(days=1))
                ultimo_texto = texto
                ultima_iso = f.isoformat() if f else ""
                if f:
                    anterior = f
            iso = ultima_iso
        else:
            # El orden no importa: cada texto se resuelve una sola vez y se guarda.
            if texto not in ya_resueltas:
                if metodo == "mes del archivo":
                    f, fuera = resolver_por_mes(texto, anio_archivo, mes_archivo)
                else:
                    f, fuera = lecturas_posibles(texto)[0], False
                ya_resueltas[texto] = (f.isoformat() if f else "", fuera)
            iso, fuera = ya_resueltas[texto]
            if fuera:
                cuenta["filas_fuera_de_mes"] += 1

        if iso == "":
            cuenta["filas_sin_fecha"] += 1
        partes[pos_fecha] = iso
        buffer.append(sep.join(partes))
        if len(buffer) >= TAM_BLOQUE:
            bloques.append(agrupar_bloque(buffer, sep, columnas))
            buffer = []

    if buffer:
        bloques.append(agrupar_bloque(buffer, sep, columnas))
    if not bloques:
        return None

    agregado = pd.concat([b[0] for b in bloques])
    nombres = pd.concat([b[1] for b in bloques])
    fila_resumen = {"anio": anio, "archivo": nombre, "metodo_fecha": metodo, **cuenta,
                    "pasajeros": int(agregado["total"].sum())}
    return agregado, nombres, fila_resumen

## 8. Procesamiento de un año completo

`procesar_anio` procesa todos los CSV del zip de un año, los junta y aplica los filtros finales:

- saca la línea `Prueba`;
- saca las fechas de otro año y las posteriores al 31/8/2026;
- deja horas y pasajeros como números enteros.

El zip de 2015 trae, además de su CSV, copias de otros años en zips internos: se ignoran, porque cada año tiene su propio zip.

Si el archivo de un año ya existe, se saltea. Para rehacerlo, hay que borrarlo.

In [8]:
def guardar_reemplazando_anio(nuevo, ruta, anio):
    """Agrega las filas de un año a un CSV de resumen, reemplazando las anteriores de ese año."""
    if os.path.exists(ruta):
        viejo = pd.read_csv(ruta)
        nuevo = pd.concat([viejo[viejo["anio"] != anio], nuevo])
    nuevo.sort_values("anio").to_csv(ruta, index=False)


def procesar_anio(carpeta_zips, anio):
    ruta_zip = os.path.join(carpeta_zips, f"molinetes-{anio}.zip")
    ruta_salida = os.path.join(CARPETA_SALIDA, f"molinetes_{anio}.csv.gz")
    if os.path.exists(ruta_salida):
        print(f"{anio}: ya procesado ({ruta_salida}), se saltea.")
        return
    if not os.path.exists(ruta_zip):
        print(f"{anio}: no se encontró {ruta_zip}, se saltea.")
        return

    print(f"\n{anio}: procesando {ruta_zip}")
    agregados, nombres, resumen = [], [], []
    with zipfile.ZipFile(ruta_zip) as zf:
        for info in zf.infolist():
            # Solo los CSV de primer nivel. El zip de 2015 trae además copias de otros
            # años dentro de zips internos: se ignoran porque cada año tiene su propio zip.
            if not info.filename.lower().endswith(".csv"):
                continue
            resultado = procesar_csv(zf, info.filename, anio)
            if resultado is None:
                continue
            agregados.append(resultado[0])
            nombres.append(resultado[1])
            resumen.append(resultado[2])
            r = resultado[2]
            print(f"  {os.path.basename(info.filename)}: {r['filas']:,} filas, "
                  f"{r['pasajeros']:,} pasajeros, sin fecha: {r['filas_sin_fecha']:,}, "
                  f"fuera del mes: {r['filas_fuera_de_mes']:,}")

    datos = pd.concat(agregados)
    datos = datos.groupby(["fecha", "hora", "linea", "estacion"], as_index=False)[COLUMNAS_PAX].sum()

    # Filtros finales: línea de prueba, fechas de otro año y fechas posteriores al último dato real.
    antes = datos["total"].sum()
    datos = datos[datos["linea"] != "LineaPRUEBA"]
    datos = datos[(datos["fecha"].dt.year == anio) & (datos["fecha"] <= ULTIMA_FECHA)]
    print(f"  Pasajeros descartados por los filtros finales: {antes - datos['total'].sum():,.0f}")

    datos["hora"] = datos["hora"].astype(int)
    for columna in COLUMNAS_PAX:
        datos[columna] = datos[columna].round().astype("int64")
    datos["fecha"] = datos["fecha"].dt.strftime("%Y-%m-%d")

    os.makedirs(CARPETA_SALIDA, exist_ok=True)
    datos.to_csv(ruta_salida, index=False, compression="gzip")
    print(f"  Guardado {ruta_salida}: {len(datos):,} filas, "
          f"{os.path.getsize(ruta_salida) / 1_000_000:.1f} MB")

    tabla_nombres = pd.concat(nombres).groupby(["linea", "estacion", "estacion_original"],
                                               as_index=False)["total"].sum()
    tabla_nombres.insert(0, "anio", anio)
    guardar_reemplazando_anio(tabla_nombres, ARCHIVO_ESTACIONES, anio)
    guardar_reemplazando_anio(pd.DataFrame(resumen), ARCHIVO_RESUMEN, anio)

## 9. Recuperar años ya procesados

Si una corrida anterior se cortó, los años que alcanzaron a terminar están en la copia de Drive. Esta celda los trae al repositorio para no repetirlos.

In [9]:
os.makedirs(CARPETA_SALIDA, exist_ok=True)
if os.path.exists(CARPETA_RESPALDO):
    for raiz, carpetas, archivos in os.walk(CARPETA_RESPALDO):
        for archivo in archivos:
            origen = os.path.join(raiz, archivo)
            destino = os.path.join("data/processed", os.path.relpath(origen, CARPETA_RESPALDO))
            if not os.path.exists(destino):
                os.makedirs(os.path.dirname(destino), exist_ok=True)
                shutil.copy(origen, destino)
                print("Recuperado:", destino)
print("Ya procesados:", sorted(os.listdir(CARPETA_SALIDA)))

Recuperado: data/processed/resumen_limpieza.csv
Recuperado: data/processed/estaciones_por_anio.csv
Recuperado: data/processed/por_anio/molinetes_2026.csv.gz
Recuperado: data/processed/por_anio/molinetes_2020.csv.gz
Recuperado: data/processed/por_anio/molinetes_2021.csv.gz
Recuperado: data/processed/por_anio/molinetes_2015.csv.gz
Recuperado: data/processed/por_anio/molinetes_2017.csv.gz
Recuperado: data/processed/por_anio/molinetes_2024.csv.gz
Recuperado: data/processed/por_anio/molinetes_2016.csv.gz
Recuperado: data/processed/por_anio/molinetes_2022.csv.gz
Recuperado: data/processed/por_anio/molinetes_2025.csv.gz
Recuperado: data/processed/por_anio/molinetes_2018.csv.gz
Recuperado: data/processed/por_anio/molinetes_2023.csv.gz
Recuperado: data/processed/por_anio/molinetes_2019.csv.gz
Ya procesados: ['molinetes_2015.csv.gz', 'molinetes_2016.csv.gz', 'molinetes_2017.csv.gz', 'molinetes_2018.csv.gz', 'molinetes_2019.csv.gz', 'molinetes_2020.csv.gz', 'molinetes_2021.csv.gz', 'molinetes_202

## 10. Procesar todos los años

Es la celda que tarda. Para cada CSV original muestra cuántas filas leyó, cuántos pasajeros sumó y cuántas filas no tenían fecha o tenían una fecha fuera del mes. Después de cada año copia `data/processed/` a Drive.

In [10]:
for anio in ANIOS:
    procesar_anio(CARPETA_ZIPS, anio)
    shutil.copytree("data/processed", CARPETA_RESPALDO, dirs_exist_ok=True)
print("\nListo. Respaldo en Drive actualizado.")

2015: ya procesado (data/processed/por_anio/molinetes_2015.csv.gz), se saltea.
2016: ya procesado (data/processed/por_anio/molinetes_2016.csv.gz), se saltea.
2017: ya procesado (data/processed/por_anio/molinetes_2017.csv.gz), se saltea.
2018: ya procesado (data/processed/por_anio/molinetes_2018.csv.gz), se saltea.
2019: ya procesado (data/processed/por_anio/molinetes_2019.csv.gz), se saltea.
2020: ya procesado (data/processed/por_anio/molinetes_2020.csv.gz), se saltea.
2021: ya procesado (data/processed/por_anio/molinetes_2021.csv.gz), se saltea.
2022: ya procesado (data/processed/por_anio/molinetes_2022.csv.gz), se saltea.
2023: ya procesado (data/processed/por_anio/molinetes_2023.csv.gz), se saltea.
2024: ya procesado (data/processed/por_anio/molinetes_2024.csv.gz), se saltea.
2025: ya procesado (data/processed/por_anio/molinetes_2025.csv.gz), se saltea.
2026: ya procesado (data/processed/por_anio/molinetes_2026.csv.gz), se saltea.

Listo. Respaldo en Drive actualizado.


## 11. Controles

Compara con los valores calculados antes para 2019, 2020, 2021, 2025 y 2026, y muestra el promedio diario y los días con datos de cada mes de todos los años.

- *Pasajeros totales* debería dar casi igual: la suma no depende de cómo se interpreten las fechas.
- Los promedios de 2025 y 2026 pueden diferir: el método anterior resolvía las fechas mirando el orden del archivo, pero en los archivos mensuales las filas vienen desordenadas. **Si algo dice REVISAR, no se ajusta nada: hay que entender la diferencia.**
- En la tabla de días, un mes con menos días que el calendario es un hueco en los datos originales.

In [11]:
def cargar_procesados(anios):
    partes = []
    for anio in anios:
        ruta = os.path.join(CARPETA_SALIDA, f"molinetes_{anio}.csv.gz")
        if os.path.exists(ruta):
            partes.append(pd.read_csv(ruta, parse_dates=["fecha"]))
    return pd.concat(partes) if partes else None


def promedio_diario(df, anio, mes):
    del_mes = df[(df["fecha"].dt.year == anio) & (df["fecha"].dt.month == mes)]
    dias = del_mes["fecha"].nunique()
    return del_mes["total"].sum() / dias if dias else float("nan")


def total_mensual(df, anio, mes):
    return df[(df["fecha"].dt.year == anio) & (df["fecha"].dt.month == mes)]["total"].sum()


def controles():
    print("\n" + "=" * 70)
    print("CONTROLES contra los valores calculados antes (2019, 2020, 2021, 2025, 2026)")
    print("=" * 70)
    df = cargar_procesados([2019, 2020, 2021, 2025, 2026])
    if df is None:
        print("Faltan años para comparar.")
    else:
        obtenidos = {
            "Pasajeros totales": df["total"].sum(),
            "Fechas distintas": df["fecha"].nunique(),
            "Promedio diario feb-2020": promedio_diario(df, 2020, 2),
            "Promedio diario abr-2020": promedio_diario(df, 2020, 4),
            "Promedio diario abr-2019": promedio_diario(df, 2019, 4),
            "Promedio diario ago-2025": promedio_diario(df, 2025, 8),
            "Promedio diario ago-2026": promedio_diario(df, 2026, 8),
            "Total ene-2019": total_mensual(df, 2019, 1),
            "Total abr-2020": total_mensual(df, 2020, 4),
        }
        for nombre, esperado in CONTROLES.items():
            obtenido = obtenidos[nombre]
            diferencia = (obtenido - esperado) / esperado * 100
            marca = "OK" if abs(diferencia) < 0.5 else "REVISAR"
            print(f"{nombre:<26} esperado {esperado:>14,.0f}   obtenido {obtenido:>14,.0f}"
                  f"   ({diferencia:+.2f}%)  {marca}")

    print("\nPROMEDIO DIARIO POR MES (miles de pasajeros) y DÍAS CON DATOS, todos los años")
    df = cargar_procesados(ANIOS)
    if df is None:
        return
    diario = df.groupby("fecha", as_index=False)["total"].sum()
    diario["anio"] = diario["fecha"].dt.year
    diario["mes"] = diario["fecha"].dt.month
    promedio = diario.pivot_table(index="anio", columns="mes", values="total", aggfunc="mean")
    dias = diario.pivot_table(index="anio", columns="mes", values="total", aggfunc="count")
    print((promedio / 1000).round(0).to_string())
    print("\nDías con datos por mes:")
    print(dias.to_string())


controles()


CONTROLES contra los valores calculados antes (2019, 2020, 2021, 2025, 2026)
Pasajeros totales          esperado    855,064,366   obtenido    855,064,601   (+0.00%)  OK
Fechas distintas           esperado          1,700   obtenido          1,700   (+0.00%)  OK
Promedio diario feb-2020   esperado        767,900   obtenido        767,869   (-0.00%)  OK
Promedio diario abr-2020   esperado         28,400   obtenido         28,430   (+0.11%)  OK
Promedio diario abr-2019   esperado        965,900   obtenido        965,933   (+0.00%)  OK
Promedio diario ago-2025   esperado        610,600   obtenido        610,572   (-0.00%)  OK
Promedio diario ago-2026   esperado        586,500   obtenido        586,520   (+0.00%)  OK
Total ene-2019             esperado     22,849,359   obtenido     22,849,359   (+0.00%)  OK
Total abr-2020             esperado        852,910   obtenido        852,910   (+0.00%)  OK

PROMEDIO DIARIO POR MES (miles de pasajeros) y DÍAS CON DATOS, todos los años
mes      1     

## 12. Revisar qué se va a subir

Solo tiene que aparecer `data/processed/`. Ningún archivo puede ser un zip ni pesar más de 50 MB.

In [12]:
!git add data/processed
!git status --short

problemas = []
for raiz, carpetas, archivos in os.walk("data/processed"):
    for archivo in archivos:
        ruta = os.path.join(raiz, archivo)
        tam_mb = os.path.getsize(ruta) / 1_000_000
        print(f"{ruta}: {tam_mb:.1f} MB")
        if archivo.endswith(".zip") or tam_mb > 50:
            problemas.append(ruta)

if problemas:
    print("\nNO COMMITEAR, revisar:", problemas)
else:
    print("\nOK: todo liviano.")

A  data/processed/estaciones_por_anio.csv
A  data/processed/por_anio/molinetes_2015.csv.gz
A  data/processed/por_anio/molinetes_2016.csv.gz
A  data/processed/por_anio/molinetes_2017.csv.gz
A  data/processed/por_anio/molinetes_2018.csv.gz
A  data/processed/por_anio/molinetes_2019.csv.gz
A  data/processed/por_anio/molinetes_2020.csv.gz
A  data/processed/por_anio/molinetes_2021.csv.gz
A  data/processed/por_anio/molinetes_2022.csv.gz
A  data/processed/por_anio/molinetes_2023.csv.gz
A  data/processed/por_anio/molinetes_2024.csv.gz
A  data/processed/por_anio/molinetes_2025.csv.gz
A  data/processed/por_anio/molinetes_2026.csv.gz
A  data/processed/resumen_limpieza.csv
data/processed/resumen_limpieza.csv: 0.0 MB
data/processed/estaciones_por_anio.csv: 0.1 MB
data/processed/.gitkeep: 0.0 MB
data/processed/por_anio/molinetes_2025.csv.gz: 5.2 MB
data/processed/por_anio/molinetes_2022.csv.gz: 5.1 MB
data/processed/por_anio/molinetes_2021.csv.gz: 3.4 MB
data/processed/por_anio/molinetes_2020.csv.gz:

## 13. Subir los datos procesados a GitHub

El token sale de los Secretos de Colab y nunca queda escrito en el notebook.

In [13]:
!git diff --cached --quiet || git commit -m "Datos procesados: agregado por fecha, hora, línea y estación (2015-2026)"

from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
url_con_token = REPO_URL.replace("https://", f"https://{token}@")

def git_con_token(argumentos):
    r = subprocess.run(["git"] + argumentos, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***"))
    return r.returncode

if git_con_token(["pull", "--rebase", url_con_token, "main"]) == 0:
    if git_con_token(["push", url_con_token, "HEAD:main"]) == 0:
        print("Listo: datos procesados subidos.")

[main b7c5fe7] Datos procesados: agregado por fecha, hora, línea y estación (2015-2026)
 14 files changed, 1319 insertions(+)
 create mode 100644 data/processed/estaciones_por_anio.csv
 create mode 100644 data/processed/por_anio/molinetes_2015.csv.gz
 create mode 100644 data/processed/por_anio/molinetes_2016.csv.gz
 create mode 100644 data/processed/por_anio/molinetes_2017.csv.gz
 create mode 100644 data/processed/por_anio/molinetes_2018.csv.gz
 create mode 100644 data/processed/por_anio/molinetes_2019.csv.gz
 create mode 100644 data/processed/por_anio/molinetes_2020.csv.gz
 create mode 100644 data/processed/por_anio/molinetes_2021.csv.gz
 create mode 100644 data/processed/por_anio/molinetes_2022.csv.gz
 create mode 100644 data/processed/por_anio/molinetes_2023.csv.gz
 create mode 100644 data/processed/por_anio/molinetes_2024.csv.gz
 create mode 100644 data/processed/por_anio/molinetes_2025.csv.gz
 create mode 100644 data/processed/por_anio/molinetes_2026.csv.gz
 create mode 100644 dat

## 14. Guardar este notebook en GitHub

Con todas las salidas a la vista: **Archivo > Guardar una copia en GitHub** → repositorio `julia-iza/Trabajo-Practico-Laboratorio`, rama `main`, ruta `notebooks/limpieza_datos.ipynb`, con la opción **Incluir un vínculo a Colab** marcada.